# Praktikum Kompresi Gambar: Lossy (SVD) & Lossless (WebP)

**Deskripsi Tugas:**
1. Algoritma: Singular Value Decomposition (SVD) untuk pendekatan kompresi *Lossy*, dan WebP Encoding untuk pendekatan *Lossless*.
2. Data Uji: Minimal 20 gambar dengan ekstensi `.webp`.
3. Output: Pengukuran metrik evaluasi (Ukuran File, Rasio Kompresi, MSE, PSNR) beserta perbandingan visual citra.

## 1. Import Library
Pustaka (library) utama yang digunakan meliputi `numpy` untuk komputasi matriks SVD, `PIL` (Pillow) untuk memanipulasi citra, `matplotlib` untuk visualisasi grafik/citra, serta `os` dan `pandas` untuk manipulasi direktori dan tabel data.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import time
import pandas as pd

## 2. Definisi Fungsi Kompresi Lossy (SVD)
Metode Singular Value Decomposition (SVD) memfaktorkan matriks citra menjadi tiga matriks komponen: $U$, $\Sigma$, dan $V^T$. Dengan mereduksi jumlah nilai singular (hanya mempertahankan $k$ nilai terbesar), citra dapat direkonstruksi dengan kebutuhan penyimpanan data yang jauh lebih kecil (merupakan sifat kompresi *Lossy*).

Karena citra uji berwarna (RGB), operasi SVD diterapkan secara terpisah pada masing-masing *channel* warna (Red, Green, Blue).

In [ ]:
def compress_image_svd(image_array, k):
    """
    Menerapkan kompresi Lossy menggunakan SVD pada citra RGB.
    Parameter 'k' merepresentasikan jumlah nilai singular (singular values) yang dipertahankan.
    """
    # Pemisahan channel warna RGB
    r = image_array[:, :, 0]
    g = image_array[:, :, 1]
    b = image_array[:, :, 2]
    
    # Fungsi internal untuk kompresi matriks 2D pada satu channel
    def compress_channel(channel, k):
        U, S, Vt = np.linalg.svd(channel, full_matrices=False)
        # Proses rekonstruksi matriks menggunakan k nilai singular utama
        reconstructed = np.dot(U[:, :k], np.dot(np.diag(S[:k]), Vt[:k, :]))
        return reconstructed
    
    r_comp = compress_channel(r, k)
    g_comp = compress_channel(g, k)
    b_comp = compress_channel(b, k)
    
    # Penggabungan kembali channel warna dan normalisasi nilai piksel ke rentang (0-255)
    compressed_img_array = np.stack((r_comp, g_comp, b_comp), axis=2)
    compressed_img_array = np.clip(compressed_img_array, 0, 255).astype('uint8')
    
    return compressed_img_array

## 3. Definisi Fungsi Kompresi Lossless (WebP)
Format WebP menyediakan fitur kompresi *lossless*, di mana algoritma ini dapat mereduksi ukuran file melalui efisiensi struktur data (*predictive coding*) tanpa membuang informasi piksel. Hal ini memastikan kualitas citra hasil kompresi 100% identik dengan citra aslinya.

In [ ]:
def compress_image_lossless_webp(image_pil, output_path):
    """
    Menerapkan kompresi Lossless dengan memanfaatkan algoritma encoding format WebP.
    """
    # Penyimpanan gambar dengan format WebP menggunakan parameter lossless=True
    image_pil.save(output_path, 'webp', lossless=True, quality=100)
    return Image.open(output_path)

## 4. Fungsi Evaluasi Metrik
Tahap ini mendefinisikan fungsi matematis untuk mengukur tingkat keberhasilan kompresi. Metrik yang dihitung mencakup Mean Squared Error (MSE), Peak Signal-to-Noise Ratio (PSNR), dan ukuran ruang penyimpanan (file size).

In [ ]:
def calculate_mse(imageA, imageB):
    err = np.sum((imageA.astype("float") - imageB.astype("float")) ** 2)
    err /= float(imageA.shape[0] * imageA.shape[1] * imageA.shape[2])
    return err

def calculate_psnr(imageA, imageB):
    mse = calculate_mse(imageA, imageB)
    if mse == 0:
        return 100 # Citra identik secara komputasional
    PIXEL_MAX = 255.0
    return 20 * np.log10(PIXEL_MAX / np.sqrt(mse))

def get_file_size(filepath):
    return os.path.getsize(filepath) / 1024 # Konversi Bytes ke KB

## 5. Inisialisasi Direktori Data
Sistem memastikan bahwa direktori kerja untuk penempatan dataset uji (`data_uji`) dan penyimpanan hasil evaluasi (`hasil_kompresi`) telah dialokasikan dengan benar.

In [ ]:
input_folder = "data_uji"
output_folder = "hasil_kompresi"

# Pembuatan direktori secara otomatis jika belum tersedia pada sistem
os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

print(f"Direktori input '{input_folder}' siap digunakan. Dataset akan dibaca dari direktori ini.")

## 6. Proses Kompresi Utama (Batch Processing)
Seluruh file berekstensi `.webp` yang berlokasi di direktori data uji akan diproses secara iteratif. Setiap citra dikenakan algoritma SVD (Lossy) dan WebP Encoding (Lossless), yang kemudian diikuti oleh pencatatan durasi eksekusi, perbandingan ukuran file, serta pengujian kualitas citra berdasarkan MSE dan PSNR.

In [ ]:
# Penentuan parameter k untuk SVD
k_svd = 50 # Jumlah nilai singular yang dipertahankan (mengendalikan trade-off kompresi vs kualitas visual)

image_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.webp')]

if len(image_files) < 20:
    print(f"Informasi: Terdapat {len(image_files)} gambar di direktori {input_folder}. Kriteria tugas menetapkan minimum 20 data uji.")

results = []

for filename in image_files:
    file_path = os.path.join(input_folder, filename)
    original_size = get_file_size(file_path)
    
    # Membaca matriks citra asli
    img_pil = Image.open(file_path).convert('RGB')
    img_array = np.array(img_pil)
    
    # 1. Proses Kompresi Lossy (Metode SVD)
    start_time = time.time()
    svd_compressed_array = compress_image_svd(img_array, k_svd)
    svd_time = time.time() - start_time
    
    # Menyimpan hasil rekonstruksi SVD ke wujud file
    svd_img_pil = Image.fromarray(svd_compressed_array)
    svd_output_path = os.path.join(output_folder, f"svd_lossy_{filename}")
    svd_img_pil.save(svd_output_path, 'webp', quality=50)
    svd_size = get_file_size(svd_output_path)
    
    # Perhitungan Metrik untuk Lossy
    svd_mse = calculate_mse(img_array, svd_compressed_array)
    svd_psnr = calculate_psnr(img_array, svd_compressed_array)
    
    # 2. Proses Kompresi Lossless (WebP Format)
    lossless_output_path = os.path.join(output_folder, f"webp_lossless_{filename}")
    start_time = time.time()
    lossless_img_pil = compress_image_lossless_webp(img_pil, lossless_output_path)
    lossless_time = time.time() - start_time
    
    lossless_size = get_file_size(lossless_output_path)
    
    # Perhitungan Metrik untuk Lossless
    lossless_img_array = np.array(lossless_img_pil)
    lossless_mse = calculate_mse(img_array, lossless_img_array)
    lossless_psnr = calculate_psnr(img_array, lossless_img_array)
    
    # Perekaman data hasil komputasi
    results.append({
        'Nama File': filename,
        'Ukuran Asli (KB)': round(original_size, 2),
        'Ukuran SVD Lossy (KB)': round(svd_size, 2),
        'SVD MSE': round(svd_mse, 2),
        'SVD PSNR (dB)': round(svd_psnr, 2),
        'Waktu SVD (s)': round(svd_time, 4),
        'Ukuran WebP Lossless (KB)': round(lossless_size, 2),
        'Lossless MSE': round(lossless_mse, 2),  # Secara teori bernilai 0
        'Lossless PSNR (dB)': round(lossless_psnr, 2), # Representasi nilai maksimal/identik
        'Waktu WebP (s)': round(lossless_time, 4)
    })

# Penyajian data evaluasi ke dalam struktur tabel interaktif
df_results = pd.DataFrame(results)
df_results

## 7. Visualisasi Perbandingan Citra
# Tampilan berikut menunjukkan komparasi visual antara citra orisinal, hasil kompresi SVD, dan hasil format WebP.

In [ ]:
if len(image_files) > 0:
    sample_file = image_files[0]
    
    original_img = Image.open(os.path.join(input_folder, sample_file)).convert('RGB')
    svd_img = Image.open(os.path.join(output_folder, f"svd_lossy_{sample_file}"))
    lossless_img = Image.open(os.path.join(output_folder, f"webp_lossless_{sample_file}"))
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(original_img)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    axes[1].imshow(svd_img)
    axes[1].set_title(f"Lossy (SVD, k={k_svd})")
    axes[1].axis('off')
    
    axes[2].imshow(lossless_img)
    axes[2].set_title("Lossless (WebP)")
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()